In [1]:
# For statistical analysis
import pandas as pd, os, datetime
import numpy as np
from scipy import stats
from scipy.stats import mannwhitneyu, iqr, skew
from scipy.signal import find_peaks

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import seaborn as sns
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
def jitter(group):
    duplicates = group.groupby(['lat', 'lon']).cumcount()
    
    # Jitter function: add a small random offset
    np.random.seed(42)  # For reproducibility
    jitter_strength = 0.05 # Adjust as needed; degrees latitude/longitude
    
    group['lat_jittered'] = group['lat'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
    group['lon_jittered'] = group['lon'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates

    return group

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_info.csv")

In [5]:
# A temporary patch to fix fuel types
gen_details['fuel_source_primary'] = gen_details['fuel_source_primary'].replace({'Solar - Solar': 'Solar','Wind - Wind': 'Wind'})
gen_details = jitter(gen_details)

In [6]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [7]:
def process_group(grp, gen_fpath, hw_tseries, start_date=None, end_date=None, mode='daily'):

    # Sanitize DUIDs and build file paths
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp, dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        raise ValueError("no files to load.")

    # Concatenate and clean header rows
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # Type conversions
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)

    # Filter by date
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index('time').sort_index()
    hw_tseries = hw_tseries.loc[start_date:end_date]

    if mode == 'daily':
        # Aggregate dfs to daily
        agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED': 'sum', 'AGCSTATUS': 'max'}
        dfs_daily = dfs.groupby(['DUID', pd.Grouper(freq='1D')]).agg(agg_func).reset_index()
        hw_tseries_daily = hw_tseries.reset_index()
        # Normalize time columns to midnight for exact matching
        dfs_daily['time'] = pd.to_datetime(dfs_daily['time']).dt.normalize()
        hw_tseries_daily['time'] = pd.to_datetime(hw_tseries_daily['time']).dt.normalize()
        # Merge on DUID and time
        merged = pd.merge(
            dfs_daily,
            hw_tseries_daily,
            on=['DUID', 'time'],
            how='left'
        )
    elif mode == 'hourly':
        dfs_hourly = dfs.reset_index()
        # Create a full hourly time index for each DUID
        duids = dfs_hourly['DUID'].unique()
        time_range = pd.date_range(start=start_date, end=end_date, freq='1h')
        full_index = pd.MultiIndex.from_product([duids, time_range], names=['DUID', 'time'])
        hw_tseries_daily = hw_tseries.reset_index().set_index(['DUID', 'time'])
        # Reindex to hourly and forward-fill
        hw_tseries_hourly = hw_tseries_daily.reindex(full_index).groupby(level=0).ffill().groupby(level=0).bfill().reset_index()
        # Merge on DUID and time
        merged = pd.merge(
            dfs_hourly,
            hw_tseries_hourly,
            on=['DUID', 'time'],
            how='left'
        )
    else:
        raise ValueError("mode must be 'daily' or 'hourly'")

    merged = merged.dropna(how='all')

    # Merge with info DataFrame
    df = merged.merge(
        grp[['DUID', 'fuel_source_primary', 'region']],
        on='DUID',
        how='left'
    )

    return df.reset_index(drop=True)

In [8]:
def select_group(gen_details, state=None, ftype=None):
    if state is not None and ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[(gen_details['region'] == state) & (gen_details['fuel_source_primary'].isin(ftype))]
        else:
            groups = gen_details.groupby(['region', 'fuel_source_primary'])
            grp = groups.get_group((state, ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[gen_details['fuel_source_primary'].isin(ftype)]
        else:
            groups = gen_details.groupby('fuel_source_primary')
            grp = groups.get_group(ftype)
    else:
        grp = gen_details

    return grp


In [9]:
sdate, edate = '2009-07-01','2024-06-30'

In [10]:
def clean_df(df, gen_details): 
    df = df.merge(gen_details[['DUID', 'reg_cap_mw','technology_type_primary','fuel_source_primary','lat_jittered','lon_jittered', 'region']], on='DUID', how='left')
    
    djf = [12, 1, 2]  # December, January, February
    df = df[df['time'].dt.month.isin(djf)]
    
    return df

In [11]:
def remove_wind_zero_rows(df, duid_col='DUID', value_col='TOTALMWh',
                          tech_col='fuel_source_primary', wind_label='Wind', threshold=5):
    """
    Remove *all rows* for wind DUIDs where the percentage of zeros 
    exceeds the given threshold. Non-wind DUIDs are left untouched.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        duid_col (str): Column name for grouping (default 'DUID')
        value_col (str): Column name with numeric values (default 'TOTALMWh')
        tech_col (str): Column that identifies technology type (default 'fuel_source_primary')
        wind_label (str): Label used for wind in tech_col (default 'Wind')
        threshold (float): Maximum allowed percentage of zeros (default 5)
    
    Returns:
        pd.DataFrame: Filtered dataframe
    """
    
    # work only on wind rows
    wind_df = df[df[tech_col] == wind_label]
    
    # calculate % of zeros per wind DUID
    percent_zeros = (
        wind_df.groupby(duid_col)[value_col]
               .apply(lambda x: (x == 0).sum() / len(x) * 100)
    )
    
    # keep DUIDs below threshold
    keep_duids = percent_zeros[percent_zeros <= threshold].index
    
    # filter wind df
    wind_filtered = wind_df[wind_df[duid_col].isin(keep_duids)]
    
    # keep all non-wind rows
    non_wind_df = df[df[tech_col] != wind_label]
    
    # combine and return
    return pd.concat([non_wind_df, wind_filtered], ignore_index=True)

In [12]:
def min_heatwave_days(df, min_days=10):
    """
    Filters the DataFrame to include only DUIDs with at least `min_days`
    of unique heatwave days (EHF_flag == 1).
    
    Parameters:
        df (pd.DataFrame): Input DataFrame with columns ['DUID', 'time', 'EHF_flag']
        min_days (int): Minimum number of unique heatwave days required

    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    df['date'] = df['time'].dt.date

    # Count unique heatwave days per DUID
    heatwave_days = (
        df[df['EHF_flag'] == 1]
        .groupby('DUID')['date']
        .nunique()
    )

    # Keep only DUIDs meeting the threshold
    valid_duids = heatwave_days[heatwave_days >= min_days].index
    
    return df[df['DUID'].isin(valid_duids)].copy()


In [13]:
info = select_group(gen_details,ftype=['Wind','Solar']).copy()
df = process_group(info, gen_fpath, hw_tseries, sdate, edate, mode='daily')
df = clean_df(df, info)
df = remove_wind_zero_rows(df, threshold=40)
df = min_heatwave_days(df,20)
df

Loaded 164 CSV file(s) out of 216 expected.


KeyError: 'fuel_source_primary'

In [ ]:
df = df.copy()
df['time'] = pd.to_datetime(df['time'])
df['date'] = df['time'].dt.date

# Count unique heatwave days per DUID
heatwave_days = (
    df[df['EHF_flag'] == 1]
    .groupby('DUID')['date']
    .nunique()
)
print(heatwave_days)

In [ ]:
# def flag_distributions(df, fuel_type=['Solar','Wind'], tie_threshold=0.5, iqr_diff_threshold=0.5, skew_diff_threshold=0.5, peak_diff_threshold=1):
#     """
#     Compute metrics for each DUID split by EHF_flag and flag distributions that differ too much.
    
#     Returns a summary DataFrame with flags.
#     """
#     df_filtered = df[df['fuel_source_primary'].isin(fuel_type)]
#     results = []

#     for duid in df_filtered['DUID'].unique():
#         duid_df = df_filtered[df_filtered['DUID'] == duid]

#         # Skip if no data
#         if duid_df.empty or 'EHF_flag' not in duid_df.columns:
#             continue

#         groups = {}
#         for ehf_flag in [0, 1]:
#             group_data = duid_df[duid_df['EHF_flag'] == ehf_flag]['TOTALMWh'].values
#             if len(group_data) == 0:
#                 continue

#             # Metrics
#             groups[ehf_flag] = {
#                 'iqr': iqr(group_data),
#                 'skew': skew(group_data),
#                 'prop_ties': np.mean(pd.Series(group_data).duplicated()),
#                 # Approximate number of peaks from histogram
#                 'n_peaks': len(find_peaks(np.histogram(group_data, bins=50, density=True)[0])[0])
#             }

#         # Skip if only one flag present
#         if len(groups) < 2:
#             continue

#         # Compare metrics
#         iqr_diff = abs(groups[0]['iqr'] - groups[1]['iqr']) / max(groups[0]['iqr'], groups[1]['iqr'])
#         skew_diff = abs(groups[0]['skew'] - groups[1]['skew'])
#         peak_diff = abs(groups[0]['n_peaks'] - groups[1]['n_peaks'])
#         max_prop_ties = max(groups[0]['prop_ties'], groups[1]['prop_ties'])

#         flag = (iqr_diff > iqr_diff_threshold) or (skew_diff > skew_diff_threshold) or \
#                (peak_diff > peak_diff_threshold) or (max_prop_ties > tie_threshold)

#         results.append({
#             'DUID': duid,
#             'iqr_diff': iqr_diff,
#             'skew_diff': skew_diff,
#             'peak_diff': peak_diff,
#             'max_prop_ties': max_prop_ties,
#             'flag_too_different': flag
#         })

#     return pd.DataFrame(results)


# flagged_df = flag_distributions(df, fuel_type=['Solar','Wind'])
# # Display only flagged DUIDs
# flagged_df[flagged_df['flag_too_different']]


In [ ]:
def plot_totalmwh_histograms_by_duid(df, fuel_type=['Solar','Wind'], col='log_t'):
    """
    Plot interactive histograms of TOTALMWh for each DUID in the given fuel type.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with columns 'DUID', 'TOTALMWh', and 'fuel_source_primary'.
        fuel_type (str): Filter rows where fuel_source_primary == fuel_type.
    """
    # Filter by fuel type
    df_filtered = df[df['fuel_source_primary'].isin(fuel_type)]

    # Get unique DUIDs
    duids = df_filtered['DUID'].unique()

    # Create figure
    fig = go.Figure()

    # Add histogram traces per DUID
    for i, duid in enumerate(duids):
        filtered_df = df_filtered[df_filtered['DUID'] == duid]
        fig.add_trace(
            go.Histogram(
                x=filtered_df[col],
                name=duid,
                visible=(i == 0)  # Show only the first DUID by default
            )
        )

    # Create dropdown buttons
    buttons = []
    for i, duid in enumerate(duids):
        visibility = [False] * len(duids)
        visibility[i] = True
        buttons.append(
            dict(
                label=duid,
                method='update',
                args=[{'visible': visibility},
                      {'title': f"Histogram of TOTALMWh for DUID: {duid}"}]
            )
        )

    # Update layout with dropdown
    fig.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=buttons,
                x=0.5,
                xanchor="center",
                y=1.15,
                yanchor="top"
            )
        ],
        title=f"Histogram of TOTALMWh for DUID: {duids[0]}",
        xaxis_title="TOTALMWh",
        yaxis_title="Count"
    )

    fig.show()

# plot_totalmwh_histograms_by_duid(df,fuel_type=['Solar'], col='TOTALMWh')

In [ ]:
def save_totalmwh_histograms_by_duid(df, fuel_type=['Solar','Wind'], output_dir="histograms"):
    """
    Generate and save histograms of TOTALMWh for each DUID in the given fuel type(s) using matplotlib.
    - Solar plotted using raw TOTALMWh
    - Wind plotted using precomputed log_t column
    """
    # Make output folder
    os.makedirs(output_dir, exist_ok=True)

    # Filter by selected fuel types
    df_filtered = df[df['fuel_source_primary'].isin(fuel_type)]

    # Loop over DUIDs
    for duid in df_filtered['DUID'].unique():
        duid_df = df_filtered[df_filtered['DUID'] == duid]

        if duid_df.empty:
            continue

        x = duid_df['TOTALMWh']
        xlabel = "TOTALMWh"

        # Plot histogram
        plt.figure(figsize=(8, 5))
        plt.hist(x, bins=50, color="skyblue", edgecolor="black")
        plt.title(f"Histogram of {xlabel} for DUID: {duid}")
        plt.xlabel(xlabel)
        plt.ylabel("Count")
        plt.grid(True, linestyle="--", alpha=0.6)

        # Save PNG
        filepath = os.path.join(output_dir, f"{duid}_histogram.png")
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        plt.close()

    print(f"Saved histograms for {len(df_filtered['DUID'].unique())} DUIDs to folder: {output_dir}")


# Example usage
# save_totalmwh_histograms_by_duid(df, fuel_type=['Solar','Wind'], output_dir="/g/data/ng72/ms5578/ID_HW_BARRA/data/output/sol_wind_hist")

In [ ]:
def qqplot_with_ci(data, dist='norm', n_sim=1000, ci=0.95, ax=None, label=None):
    data = np.sort(data.dropna())
    n = len(data)
    probs = (np.arange(1, n + 1) - 0.5) / n  # plotting positions

    # Fit parameters for the chosen distribution
    if dist == 'norm':
        mu, sigma = np.mean(data), np.std(data, ddof=1)
        theoretical_quantiles = stats.norm.ppf(probs, loc=mu, scale=sigma)
    else:
        raise ValueError("Only 'norm' distribution is supported in this function.")

    # Simulate n_sim samples from fitted normal distribution and compute quantiles
    sim_q = np.array([np.sort(np.random.normal(loc=mu, scale=sigma, size=n)) for _ in range(n_sim)])
    lower_bound = np.percentile(sim_q, (1 - ci) / 2 * 100, axis=0)
    upper_bound = np.percentile(sim_q, (1 + ci) / 2 * 100, axis=0)

    # Plotting
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    ax.fill_between(theoretical_quantiles, lower_bound, upper_bound,
                    color='lightgray', label=f'{int(ci*100)}% CI')
    ax.plot(theoretical_quantiles, data, 'o', label='Data')
    ax.plot(theoretical_quantiles, theoretical_quantiles, 'r--', label='Ideal Fit')

    title = "Q-Q Plot with Confidence Interval"
    if label is not None:
        title += f" — {label}"
    ax.set_title(title)

    ax.set_xlabel("Theoretical Quantiles (Normal)")
    ax.set_ylabel("Sample Quantiles")
    ax.legend()
    ax.grid(True)

    return ax

# # Example: Apply to a single DUID, using raw data
# duid = 'BANGOWF1'
# sample = df[df['DUID'] == duid]['TOTALMWh']
# qqplot_with_ci(sample, label=duid)
# plt.show()

In [ ]:
def save_qqplots_by_duid(df, fuel_type=['Solar','Wind'], output_dir="qqplots"):
    """
    Generate and save Q-Q plots for each DUID:
    - Solar uses raw TOTALMWh
    - Wind uses precomputed log_t
    """
    os.makedirs(output_dir, exist_ok=True)

    # Filter data
    df_filtered = df[df['fuel_source_primary'].isin(fuel_type)]

    # Loop over DUIDs
    for duid in df_filtered['DUID'].unique():
        duid_df = df_filtered[df_filtered['DUID'] == duid]

        if duid_df.empty:
            continue
            
        sample = duid_df['TOTALMWh']
        xlabel = "TOTALMWh"

        # Plot
        fig, ax = plt.subplots(figsize=(6, 6))
        qqplot_with_ci(sample, label=f"{duid} ({xlabel})", ax=ax)

        # Save PNG
        filepath = os.path.join(output_dir, f"{duid}_qqplot.png")
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved Q-Q plots for {len(df_filtered['DUID'].unique())} DUIDs to folder: {output_dir}")


# Example usage
# save_qqplots_by_duid(df, fuel_type=['Solar','Wind'], output_dir="/g/data/ng72/ms5578/ID_HW_BARRA/data/output/sol_wind_hist_qqplots")

In [ ]:
def save_boxplots_by_duid_ehf(df, fuel_type=['Solar','Wind'], output_dir="boxplots"):
    """
    Generate and save boxplots for each DUID split by EHF_flag (0 vs 1):
    - Boxplots help check distribution shape and outliers.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Filter data
    df_filtered = df[df['fuel_source_primary'].isin(fuel_type)]

    # Loop over DUIDs
    for duid in df_filtered['DUID'].unique():
        duid_df = df_filtered[df_filtered['DUID'] == duid]

        if duid_df.empty:
            continue

        # Make sure EHF_flag exists and has both 0 and 1 if present
        ehf_values = duid_df['EHF_flag'].unique()
        if len(ehf_values) == 0:
            continue

        # Plot
        fig, ax = plt.subplots(figsize=(6, 6))
        sns.boxplot(x='EHF_flag', y='TOTALMWh', data=duid_df, ax=ax)
        ax.set_title(f"Boxplot for {duid} by EHF_flag")
        ax.set_xlabel("EHF_flag (0=No, 1=Yes)")
        ax.set_ylabel("TOTALMWh")

        # Save PNG
        filepath = os.path.join(output_dir, f"{duid}_boxplot_ehf.png")
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved boxplots by EHF_flag for {len(df_filtered['DUID'].unique())} DUIDs to folder: {output_dir}")

# Example usage:
# save_boxplots_by_duid_ehf(df, fuel_type=['Solar','Wind'], output_dir="/g/data/ng72/ms5578/ID_HW_BARRA/data/output/boxplot")


In [ ]:
def save_violinplots_by_duid_ehf(df, fuel_type=['Solar','Wind'], output_dir="violinplots", overlay_swarm=True):
    """
    Generate and save violin plots for each DUID split by EHF_flag (0 vs 1):
    - Violin plots show distribution density and quartiles.
    - Optionally overlay individual points with a swarm plot.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Filter data by fuel type
    df_filtered = df[df['fuel_source_primary'].isin(fuel_type)]

    # Loop over each DUID
    for duid in df_filtered['DUID'].unique():
        duid_df = df_filtered[df_filtered['DUID'] == duid].copy()

        if duid_df.empty:
            continue

        # Ensure EHF_flag is str
        duid_df['EHF_flag'] = duid_df['EHF_flag'].astype(str)

        # Plot
        fig, ax = plt.subplots(figsize=(6, 6))

        # Palette for numeric EHF_flag
        palette = {'0.0': "skyblue", '1.0': "orange"}

        sns.violinplot(x=None, y='TOTALMWh', hue='EHF_flag', data=duid_df, palette=palette, inner='quartile', legend=False)


        if overlay_swarm:
            sns.stripplot(
                x='EHF_flag',
                y='TOTALMWh',
                data=duid_df,
                color='k',
                alpha=0.5,
                jitter=0.25,  # controls horizontal spread
                size=2,
                ax=ax
)

        ax.set_title(f"Violin Plot for {duid} by EHF_flag")
        ax.set_xlabel("EHF_flag (0=No, 1=Yes)")
        ax.set_ylabel("TOTALMWh")

        # Save PNG
        filepath = os.path.join(output_dir, f"{duid}_violinplot_ehf.png")
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved violin plots by EHF_flag for {len(df_filtered['DUID'].unique())} DUIDs to folder: {output_dir}")


save_violinplots_by_duid_ehf(df, fuel_type=['Solar','Wind'], output_dir="/g/data/ng72/ms5578/ID_HW_BARRA/data/output/violinplot")


In [ ]:
def mannwhitu_hw(df, var):
    """
    For each DUID, compare a variable (e.g., log_t for Wind) between EHF_flag==0 and EHF_flag==1 using mannwhitu's t-test.
    Optionally, return group means in the original variable (e.g., TOTALMWh).
    
    Parameters:
        df (pd.DataFrame): DataFrame with 'DUID', 'EHF_flag', and variable columns.
        var (str): Column to use for t-test (e.g., 'log_t').
        original_var (str, optional): Column to compute means in original scale (e.g., 'TOTALMWh').
        
    Returns:
        pd.DataFrame with DUID, t-statistic, p-value, and group means.
    """
    results = []

    for duid, group in df.groupby('DUID'):
        # t-test variable
        group_0 = group[group['EHF_flag'] == 0][var].dropna()
        group_1 = group[group['EHF_flag'] == 1][var].dropna()

        # Skip if group too small
        if len(group_0) < 10 or len(group_1) < 10:
            continue

        # Mann-Whitney U
        stat, p_val = mannwhitneyu(group_0, group_1, alternative='two-sided')

        mean_baseline = group_0.mean()
        mean_hw = group_1.mean()

        results.append({
            'DUID': duid,
            'statistic': stat,
            'p_value': p_val,
            'mean_baseline': mean_baseline,
            'mean_hw': mean_hw
        })

    return pd.DataFrame(results)

In [ ]:
solar_results = mannwhitu_hw(df[df['fuel_source_primary']=='Solar'].copy(), 'TOTALMWh')
wind_results = mannwhitu_hw(df[df['fuel_source_primary']=='Wind'].copy(), var='TOTALMWh')
mannwhitu_result = pd.concat([solar_results, wind_results], ignore_index=True)

mannwhitu_result[mannwhitu_result['p_value'] < 0.05].merge(info[['DUID','fuel_source_primary']],on='DUID')

In [ ]:
def plot_mannwhitu_pvalues_map(df, info, mannwhitu_results, alpha=0.05,ftypes='Wind and Solar'):
    merged = mannwhitu_results.merge(info, on='DUID', how='left').dropna(subset=['lat_jittered', 'lon_jittered'])
    merged['change'] = merged['mean_hw'] - merged['mean_baseline']
    merged['size'] = (merged['change'].abs().replace(0, 0.001) * 100) + 20

    def pvalue_significance(p):
        if p < alpha / 10:
            return 'Very strong'
        elif p < alpha:
            return 'Strong'
        elif p < 0.1:
            return 'Weak'
        else:
            return 'Not significant'

    merged['SignificanceLevel'] = merged['p_value'].apply(pvalue_significance)

    def category(row):
        sig = row['SignificanceLevel']
        if sig == 'Not significant':
            return sig
        elif row['change'] < 0:
            return f'{sig} decrease'
        else:
            return f'{sig} increase'

    merged['Category'] = merged.apply(category, axis=1)

    color_map = {
        'Very strong increase': 'darkgreen',
        'Strong increase': 'green',
        'Weak increase': 'lightgreen',
        'Very strong decrease': 'darkred',
        'Strong decrease': 'red',
        'Weak decrease': 'salmon',
        'Not significant': 'lightgray'
    }

    fig = px.scatter_map(
        merged,
        lat='lat_jittered',
        lon='lon_jittered',
        size='size',
        color='Category',
        color_discrete_map=color_map,
        hover_name='DUID',
        hover_data={
            'change': ':.2f',
            'mean_baseline': ':.2f',
            'mean_hw': ':.2f',
            'p_value': ':.4f'
        },
        zoom=5,
        map_style='carto-darkmatter',
        title=f'Shift in Generation Distribution on Heatwave days for {ftypes}'
    )

    fig.update_layout(
        legend_title_text='Significance and Direction',
        margin=dict(l=10, r=10, t=50, b=10)
    )

    fig.show()
    # fig.write_html("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/mannwhitu.html")


In [ ]:
plot_mannwhitu_pvalues_map(df, info, mannwhitu_result, alpha=0.05)

In [ ]:
plot_mannwhitu_pvalues_map(df, info, wind_results, alpha=0.05,ftypes='Wind')

In [ ]:
# yearly = df.copy()
# yearly['year'] = pd.DatetimeIndex(yearly['time']).year
# yearly['norm_std'] = yearly.groupby(['DUID','year'])['TOTALMWh'].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
# yearly = yearly.pivot_table(index=['DUID','EHF_flag'], columns='year', values='TOTALMWh', aggfunc='mean')
# yearly.loc['WDGPH1']

In [ ]:
# count_spell = df.copy()
# count_spell['year'] = pd.to_datetime(count_spell['time']).dt.year

# # Ensure data is sorted so spells are detected correctly
# count_spell = count_spell.sort_values(['DUID', 'time'])

# # Function to count heatwave spells in a series
# def count_hw_spells(series):
#     # A spell starts where current is 1 and previous is not 1
#     return ((series == 1) & (series.shift() != 1)).sum()

# # Apply per DUID-year group
# count_spell = (
#     count_spell
#     .groupby(['DUID', 'year'])['EHF_flag']
#     .apply(count_hw_spells)
#     .reset_index(name='HW_spells')
# )

# count_spell[count_spell['DUID'] == 'BOCORWF1']

In [ ]:
# count_days = df.copy()
# count_days['year'] = pd.DatetimeIndex(count_days['time']).year
# count_days = count_days.pivot_table(index=['year'], columns='DUID', values='EHF_flag', aggfunc='sum')
# count_days

In [ ]:
def boxplot_buckets(df, value_col='TOTALMWh', event_day_col='HW_event_day',
                    category_col=None, category_value=None):
    df_plot = df.copy()

    # Optional category filtering
    if category_col and category_value:
        df_plot = df_plot[df_plot[category_col] == category_value]

    # Create buckets: 1,2,3,last
    df_plot['bucket'] = df_plot[event_day_col].apply(
        lambda x: 'NaN' if pd.isna(x) else ('last' if x > 3 else str(int(x)))
    )

    # Keep only relevant buckets
    df_plot = df_plot[df_plot['bucket'].isin(['1','2','3','last'])]

    # Create box plot
    fig = px.box(
        df_plot,
        x='bucket',
        y=value_col,
        points='all',
        color='bucket',
        hover_data=['DUID'],
        title=f"{value_col} distribution per event bucket" + (f" for {category_value}" if category_value else "")
    )
    fig.show()

boxplot_buckets(df, category_col='fuel_source_primary', value_col='TOTALMWh', category_value='Solar')